# 01 – Dati e verifica (Step S2)

**Obiettivo (GUIDA_UNICA §4.2, S2):**
1. Leggere `trajectories.csv` con `header=None, index_col=False` (come il professore).
2. Verificare shape `(400400, 56)`, assenza di NaN, tempo che riparte da 0 ogni 1001 righe e cresce a passi di 0.02, 400 traiettorie.
3. Scartare il tempo e fare il reshape in `(400, 1001, 55)`.
4. Statistiche per variabile (min, max, media, std) e verifica del range [-1, 1].
5. Grafici di esempio di magnetizzazioni e correlazioni.
6. Split per traiettoria (seed 42) e salvataggio di `results/split_indices.npz`.
7. Confronto rapido dati grezzi vs z-score su un modello piccolo per decidere la normalizzazione.

> **Nota sull'ordine:** lo split (punto 6) viene fatto **prima** del confronto sulla normalizzazione, così le statistiche dello z-score si calcolano solo sul train e il confronto si valuta sul validation, senza leakage.

**Gate S2:** tutte le verifiche passano (o ogni anomalia è documentata) e gli indici sono salvati.

## 0. Setup (cella comune a tutti i notebook)

In [ ]:
import os
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    
    # Risoluzione percorso del repository su Drive (gestisce cartella condivisa o nomi personalizzati)
    CANDIDATES = [
        Path("/content/drive/MyDrive/DL-Quantum"),
        Path("/content/drive/MyDrive/Quantum-Dynamics"),
        Path("/content/drive/MyDrive/Quantum Dynamics"),
    ]
    REPO_PATH = next((p for p in CANDIDATES if p.exists()), None)
    if REPO_PATH is None:
        found = list(Path("/content/drive/MyDrive").glob("**/DL-Quantum"))
        REPO_PATH = found[0] if found else Path("/content/drive/MyDrive/DL-Quantum")
else:
    # in locale il notebook vive in <repo>/notebooks
    REPO_PATH = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

# Nome breve di chi esegue (separa i results_log su Drive: es. 'riccardo' o 'collega')
os.environ.setdefault("DLQ_AUTHOR", "riccardo")
os.environ.setdefault("DLQ_ROOT", str(REPO_PATH))

from src.seeding import set_seeds
SEED = 42
set_seeds(SEED)

try:
    import tensorflow as tf
    print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
except ImportError:
    tf = None
    print("TensorFlow non installato (ok in locale: serve solo per il punto 7)")

print("Repo:", REPO_PATH, "| Colab:", IN_COLAB)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.config import (CORR_SLICE, DT, MAG_SLICE, N_FEATURES, N_POINTS, N_QUBITS,
                        N_TRAJ, get_paths, make_config)
from src.data import (corr_pairs, feature_names, feature_stats, load_raw_csv, load_split,
                      make_split, save_split, to_array, validate_raw)
from src.plots import plot_trajectory, save_figure

CFG = make_config()
PATHS = get_paths()
NAMES = feature_names()
for k, v in PATHS.items():
    print(f"{k:12s} -> {v}")

if not PATHS["data_csv"].exists():
    print("\n[ATTENZIONE] trajectories.csv non trovato nel percorso predefinito!")
    print("Se la cartella è condivisa su Drive con il collega, assicurati che il collega abbia aggiunto")
    print("la scorciatoia in 'Il mio Drive' (Tasto destro -> Organizza -> Aggiungi scorciatoia -> Il mio Drive)")
    print("Oppure imposta os.environ['DLQ_DATA_CSV'] = '/percorso/effettivo/trajectories.csv'")
assert PATHS["data_csv"].exists(), f"CSV non trovato: {PATHS['data_csv']}"
print("\nDataset trajectories.csv trovato con successo! ✔")

## 1. Lettura del CSV

Il file è di circa 464 MB: la lettura richiede qualche decina di secondi.

In [ ]:
t0 = time.time()
df = load_raw_csv(PATHS["data_csv"])
print(f"Letto in {time.time() - t0:.1f} s | shape = {df.shape}")
df.head()

## 2. Verifiche di integrità

Il professore scrive: *"Code provided by the teacher is just an example. Correctness is your responsibility"*. Verifichiamo:
- shape `(400400, 56)`;
- nessun NaN;
- 400 inizi di traiettoria (`t = 0`);
- griglia temporale regolare `t = k · 0.02`, `k = 0..1000`, in ogni traiettoria;
- range dei valori in [-1, 1].

In [ ]:
report = validate_raw(df)
display(pd.Series(report, name="valore").to_frame())

# Controlli aggiuntivi sul tempo
t = df.iloc[:, 0].to_numpy(dtype=np.float64)
starts = np.flatnonzero(np.isclose(t, 0.0, atol=1e-6))
print("Inizi traiettoria attesi ogni 1001 righe:",
      np.array_equal(starts, np.arange(N_TRAJ) * N_POINTS))
dt_all = np.diff(t.reshape(N_TRAJ, N_POINTS), axis=1)
print(f"dt: min = {dt_all.min():.8f}, max = {dt_all.max():.8f} (atteso {DT})")
print(f"t finale per traiettoria: {np.unique(np.round(t.reshape(N_TRAJ, N_POINTS)[:, -1], 6))}")

### 2.1 Coerenza fisica dell'ordine delle correlazioni

Il README dice che le correlazioni sono in ordine `c12..c1N, c23..c2N, ..., c(N-1)N`. Lo verifichiamo con un vincolo fisico. Per due spin con autovalori ±1, le probabilità congiunte di misura in base z sono

$$P(s_i, s_j) = \tfrac14\,\big(1 + s_i\,m_i + s_j\,m_j + s_i s_j\,c_{ij}\big) \ge 0, \qquad s_i, s_j \in \{\pm 1\}.$$

Se l'ordine delle colonne fosse sbagliato (correlazione associata alla coppia sbagliata), questi vincoli verrebbero violati. Controlliamo anche lo stato iniziale: se a `t = 0` lo stato è prodotto in base z, vale $c_{ij}(0) = m_i(0)\,m_j(0)$.

In [ ]:
data = to_array(df)                     # (400, 1001, 55), tempo scartato
assert data.shape == (N_TRAJ, N_POINTS, N_FEATURES), data.shape
print("data:", data.shape, data.dtype)

pairs = np.array(corr_pairs()) - 1      # 0-based
mag = data[..., MAG_SLICE].astype(np.float64)
cor = data[..., CORR_SLICE].astype(np.float64)
mi, mj = mag[..., pairs[:, 0]], mag[..., pairs[:, 1]]

probs = np.stack([(1 + si * mi + sj * mj + si * sj * cor) / 4
                  for si in (1, -1) for sj in (1, -1)])
min_prob = probs.min()
tol = 1e-4
n_viol = int((probs < -tol).sum())
print(f"Probabilità congiunta minima: {min_prob:.2e} | violazioni (< -{tol}): {n_viol}")

# Stato iniziale prodotto?
dev0 = np.abs(cor[:, 0] - mi[:, 0] * mj[:, 0])
print(f"|c_ij(0) - m_i(0) m_j(0)|: max = {dev0.max():.2e}, media = {dev0.mean():.2e}")
print("Valori distinti di m_i(0):", np.unique(np.round(mag[:, 0], 4))[:10])

# Controprova: con un ordine permutato i vincoli dovrebbero rompersi più spesso
rng = np.random.default_rng(0)
perm = rng.permutation(len(pairs))
probs_perm = np.stack([(1 + si * mi + sj * mj + si * sj * cor[..., perm]) / 4
                       for si in (1, -1) for sj in (1, -1)])
print(f"Controprova (ordine permutato): violazioni = {int((probs_perm < -tol).sum())}")
del probs, probs_perm

## 3. Cache `.npy`

Salviamo l'array `(400, 1001, 55)` in `data/cache/trajectories.npy` così i notebook successivi lo caricano in un secondo con `src.data.load_trajectories()`.

In [ ]:
cache_path = PATHS["cache"] / "trajectories.npy"
np.save(cache_path, data)
print(f"Salvato {cache_path} ({cache_path.stat().st_size / 1e6:.0f} MB)")
assert np.array_equal(np.load(cache_path), data)
del df   # libera memoria: da qui in poi si usa solo `data`

## 4. Statistiche per variabile

In [ ]:
stats = feature_stats(data)
stats["range"] = stats["max"] - stats["min"]
pd.set_option("display.max_rows", 60)
display(stats.round(4))

print("Tutte le variabili in [-1, 1]:", bool((stats["min"] >= -1).all() and (stats["max"] <= 1).all()))
print("\nRiepilogo per gruppo:")
grp = pd.DataFrame({
    name: {"min": data[..., sl].min(), "max": data[..., sl].max(),
           "mean": data[..., sl].mean(), "std": data[..., sl].std()}
    for name, sl in [("magnetizzazioni", MAG_SLICE), ("correlazioni", CORR_SLICE)]
}).T
display(grp.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5), gridspec_kw={"width_ratios": [1, 3]})
x = np.arange(N_FEATURES)
for ax, sl, title in [(axes[0], MAG_SLICE, "Magnetizzazioni"), (axes[1], CORR_SLICE, "Correlazioni")]:
    s = stats.iloc[sl]
    xs = x[sl]
    ax.bar(xs, s["max"] - s["min"], bottom=s["min"], color="lightsteelblue", label="[min, max]")
    ax.errorbar(xs, s["mean"], yerr=s["std"], fmt="o", ms=3, color="navy", label="media ± std")
    ax.set_xticks(xs)
    ax.set_xticklabels(s.index, rotation=90, fontsize=6)
    ax.axhline(1, color="gray", ls=":", lw=0.8)
    ax.axhline(-1, color="gray", ls=":", lw=0.8)
    ax.set_title(title)
axes[0].legend(fontsize=7)
fig.tight_layout()
print(save_figure(fig, "s2_feature_stats", subdir="01_data"))
plt.show()

In [ ]:
# Distribuzione dei valori e std nel tempo (quanto è "viva" la dinamica)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(data[..., MAG_SLICE].ravel(), bins=100, alpha=0.7, density=True, label="magnetizzazioni")
axes[0].hist(data[..., CORR_SLICE].ravel(), bins=100, alpha=0.7, density=True, label="correlazioni")
axes[0].set_title("Distribuzione dei valori")
axes[0].legend()

tt = np.arange(N_POINTS) * DT
axes[1].plot(tt, data[..., MAG_SLICE].std(axis=(0, 2)), label="magnetizzazioni")
axes[1].plot(tt, data[..., CORR_SLICE].std(axis=(0, 2)), label="correlazioni")
axes[1].set_xlabel("t")
axes[1].set_title("Std tra traiettorie e variabili nel tempo")
axes[1].legend()
fig.tight_layout()
print(save_figure(fig, "s2_value_distributions", subdir="01_data"))
plt.show()

## 5. Grafici di esempio

Tre traiettorie: tutte le magnetizzazioni e alcune correlazioni (primi vicini `c1_2`, `c5_6` e lungo raggio `c1_10`).

In [ ]:
example_traj = [0, 1, 2]
corr_examples = [NAMES.index(n) for n in ("c1_2", "c5_6", "c1_10")]

fig, axes = plt.subplots(len(example_traj), 2, figsize=(14, 2.8 * len(example_traj)), sharex=True)
for r, tid in enumerate(example_traj):
    plot_trajectory(data[tid], list(range(N_QUBITS)), ax=axes[r, 0],
                    title=f"Traiettoria {tid} – magnetizzazioni")
    plot_trajectory(data[tid], corr_examples, ax=axes[r, 1],
                    title=f"Traiettoria {tid} – correlazioni")
    for ax in axes[r]:
        ax.legend(ncol=5, fontsize=6, loc="upper right")
fig.tight_layout()
print(save_figure(fig, "s2_example_trajectories", subdir="01_data"))
plt.show()

In [ ]:
# Zoom sui primi 2 tempi: scala della dinamica rispetto a T_out = 40 step (0.8 unità)
tid = example_traj[0]
fig, ax = plt.subplots(figsize=(9, 3))
plot_trajectory(data[tid, :101], [0, 1, NAMES.index("c1_2")], ax=ax,
                title=f"Traiettoria {tid} – primi 100 step (t ≤ 2)")
for k, lab in [(10, "T_in=10"), (20, "T_in=20"), (50, "T_in=10 + T_out=40")]:
    ax.axvline(k * DT, color="gray", ls=":", lw=0.8)
    ax.text(k * DT, ax.get_ylim()[1], lab, fontsize=7, rotation=90, va="top", ha="right")
fig.tight_layout()
print(save_figure(fig, "s2_zoom_first_steps", subdir="01_data"))
plt.show()

## 6. Split per traiettoria

Lo split è **per traiettoria** (280 / 60 / 60, `split_seed = 42`): nessuna finestra di validation o test proviene da una traiettoria vista in training. Gli indici si salvano **una volta sola** e tutti i notebook successivi li ricaricano con `load_split()`.

In [ ]:
split_path = PATHS["results"] / "split_indices.npz"
new_split = make_split(n_traj=N_TRAJ, n_train=CFG["n_train"], n_val=CFG["n_val"],
                       n_test=CFG["n_test"], seed=CFG["split_seed"])

if split_path.exists():
    split = load_split(split_path)
    same = all(np.array_equal(split[k], new_split[k]) for k in new_split)
    print(f"Split già presente in {split_path} | identico a quello rigenerato: {same}")
    assert same, "Lo split salvato non coincide con make_split(seed=42): NON sovrascriverlo, indagare."
else:
    split = new_split
    print("Salvato:", save_split(split, split_path))

for k, v in split.items():
    print(f"{k:5s}: {len(v):3d} traiettorie | prime: {v[:8]}")

all_idx = np.concatenate(list(split.values()))
assert len(np.unique(all_idx)) == N_TRAJ == len(all_idx), "Split non è una partizione di 0..399"
assert not (set(split["train"]) & set(split["val"]) or set(split["train"]) & set(split["test"])
            or set(split["val"]) & set(split["test"]))
print("Partizione disgiunta e completa ✔")

In [ ]:
# Le tre partizioni hanno distribuzioni simili? (controllo di rappresentatività)
rows = []
for k, idx in split.items():
    d = data[idx]
    rows.append({"split": k,
                 "mean_mag": d[..., MAG_SLICE].mean(), "std_mag": d[..., MAG_SLICE].std(),
                 "mean_corr": d[..., CORR_SLICE].mean(), "std_corr": d[..., CORR_SLICE].std()})
display(pd.DataFrame(rows).set_index("split").round(4))

## 7. Decisione sulla normalizzazione: dati grezzi vs z-score

Tutti i valori stanno già in [-1, 1] (punto 4), quindi la scelta naturale è **dati grezzi + output `tanh`**. La guida chiede però di motivarla con un confronto rapido. Setup:

| Voce | Valore |
|---|---|
| Modello | LSTM diretto (`LSTM(64) → Dense(T_out·55) → Reshape`), lo stesso della baseline S4 |
| Finestra | `T_in = 10`, `T_out = 40`, `stride = 10` |
| Dati | sottoinsieme di 100 traiettorie di train, validation completo (60) |
| Varianti | `none`: input grezzi, output `tanh` · `zscore`: media/std per feature **dal solo train**, output lineare |
| Metrica | MSE sul validation **in scala originale** (la z-score viene invertita prima di valutare) |
| Repliche | seed 42, 43, 44 → media ± CI 95% (t di Student) |

Richiede TensorFlow (Colab). In locale la cella viene saltata.

In [ ]:
from src.windowing import make_windows
from src.metrics import evaluate
from src.stats import summarize

NORM_N_TRAIN = 100      # traiettorie di train usate nel confronto
NORM_EPOCHS = 15
NORM_SEEDS = [42, 43, 44]
T_IN, T_OUT, STRIDE = CFG["windows"]["short"], CFG["t_out"], CFG["stride"]

sub_tr = split["train"][:NORM_N_TRAIN]
X_tr, Y_tr, _ = make_windows(data, sub_tr, T_IN, T_OUT, STRIDE)
X_va, Y_va, _ = make_windows(data, split["val"], T_IN, T_OUT, STRIDE)
print("train:", X_tr.shape, Y_tr.shape, "| val:", X_va.shape, Y_va.shape)

# Statistiche z-score SOLO dal train completo (niente leakage da val/test)
flat_tr = data[split["train"]].reshape(-1, N_FEATURES)
MU = flat_tr.mean(axis=0).astype(np.float32)
SD = flat_tr.std(axis=0).astype(np.float32)
SD = np.where(SD < 1e-8, 1.0, SD).astype(np.float32)
del flat_tr

In [ ]:
def build_direct_lstm(t_in, t_out, n_feat, units=64, out_act="tanh"):
    inp = tf.keras.Input(shape=(t_in, n_feat))
    h = tf.keras.layers.LSTM(units)(inp)
    out = tf.keras.layers.Dense(t_out * n_feat, activation=out_act)(h)
    out = tf.keras.layers.Reshape((t_out, n_feat))(out)
    return tf.keras.Model(inp, out, name=f"lstm_direct_{out_act}")


NORM_VARIANTS = {
    "none":   dict(fwd=lambda a: a,               inv=lambda a: a,           act="tanh"),
    "zscore": dict(fwd=lambda a: (a - MU) / SD,   inv=lambda a: a * SD + MU, act="linear"),
}

norm_rows, norm_hist = [], {}
if tf is None:
    print("TensorFlow assente: confronto saltato (eseguire su Colab).")
else:
    for name, v in NORM_VARIANTS.items():
        for seed in NORM_SEEDS:
            set_seeds(seed)
            model = build_direct_lstm(T_IN, T_OUT, N_FEATURES, out_act=v["act"])
            model.compile(optimizer=tf.keras.optimizers.Adam(CFG["lr"]), loss="mse")
            t0 = time.time()
            h = model.fit(v["fwd"](X_tr), v["fwd"](Y_tr),
                          validation_data=(v["fwd"](X_va), v["fwd"](Y_va)),
                          epochs=NORM_EPOCHS, batch_size=CFG["batch_size"], shuffle=True, verbose=0)
            pred = v["inv"](model.predict(v["fwd"](X_va), batch_size=256, verbose=0))
            m = evaluate(Y_va, pred)
            norm_rows.append({"normalization": name, "seed": seed, **m,
                              "train_time_s": time.time() - t0})
            norm_hist[(name, seed)] = h.history
            print(f"{name:6s} seed {seed}: val MSE (scala originale) = {m['mse']:.5f} "
                  f"| mag {m['mse_mag']:.5f} | corr {m['mse_corr']:.5f}")
            tf.keras.backend.clear_session()

In [ ]:
if norm_rows:
    norm_df = pd.DataFrame(norm_rows)
    norm_df.to_csv(PATHS["results"] / "s2_normalization_comparison.csv", index=False)

    summary = []
    for name, g in norm_df.groupby("normalization"):
        row = {"normalization": name}
        for m in ["mse", "mse_mag", "mse_corr", "mae"]:
            s = summarize(g[m])
            row[m] = f"{s['mean']:.5f} ± {s['ci_half']:.5f}"
        summary.append(row)
    display(pd.DataFrame(summary).set_index("normalization"))

    # Curve di loss (attenzione: la loss z-score è in scala normalizzata, non confrontabile in valore assoluto)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    for ax, name in zip(axes, NORM_VARIANTS):
        for seed in NORM_SEEDS:
            hh = norm_hist[(name, seed)]
            ax.plot(hh["loss"], color="C0", alpha=0.6, label="train" if seed == NORM_SEEDS[0] else None)
            ax.plot(hh["val_loss"], color="C1", alpha=0.6, label="val" if seed == NORM_SEEDS[0] else None)
        ax.set_title(f"normalization = {name} (loss nella scala del modello)")
        ax.set_xlabel("epoca")
        ax.set_yscale("log")
        ax.legend()
    fig.tight_layout()
    print(save_figure(fig, "s2_normalization_loss_curves", subdir="01_data"))
    plt.show()

    means = norm_df.groupby("normalization")["mse"].mean()
    ci = {n: summarize(g["mse"]) for n, g in norm_df.groupby("normalization")}
    overlap = not (ci["none"]["ci_high"] < ci["zscore"]["ci_low"] or ci["zscore"]["ci_high"] < ci["none"]["ci_low"])
    best = means.idxmin()
    print(f"Migliore in media: {best} | CI 95% sovrapposti: {overlap}")
    if overlap:
        print("→ Differenza non significativa: si preferisce 'none' (più semplice, output tanh vincolato al range fisico).")
    else:
        print(f"→ Differenza significativa a favore di '{best}': aggiornare CFG['normalization'] in src/config.py.")

## 8. Riepilogo Gate S2

In [ ]:
gate = {
    "shape (400400, 56)": report["shape_ok"],
    "nessun NaN": report["no_nan"],
    "400 traiettorie (t=0)": report["n_traj_ok"],
    "griglia temporale dt=0.02": report["time_grid_ok"],
    "valori in [-1, 1]": report["in_unit_range"],
    "vincoli fisici correlazioni": n_viol == 0,
    "reshape (400, 1001, 55)": data.shape == (N_TRAJ, N_POINTS, N_FEATURES),
    "cache .npy salvata": cache_path.exists(),
    "split salvato e disgiunto": split_path.exists(),
    "confronto normalizzazione eseguito": bool(norm_rows),
}
for k, v in gate.items():
    print(f"{'✔' if v else '✘'} {k}")
print("\nGATE S2:", "SUPERATO" if all(gate.values()) else "da completare / documentare le anomalie nel decision_log.md")

## 9. Da riportare nel `decision_log.md`

Dopo l'esecuzione su Colab, copiare nel log (sezione **S2**):
- esito di ogni verifica (tabella del punto 8) ed eventuali anomalie con la loro spiegazione;
- range osservato di magnetizzazioni e correlazioni e se lo stato iniziale è prodotto;
- esito del confronto `none` vs `zscore` (media ± CI 95%) e decisione finale su `CFG["normalization"]`;
- conferma che `results/split_indices.npz` è stato generato con `split_seed = 42` e non va più rigenerato.